In [1]:
from dotenv import load_dotenv
import os
import sys

load_dotenv()
# set the python path
sys.path.append(os.getenv('PYTHON_PATH'))

In [2]:
from slides.scratch.reduce_final import reduce_docs
from slides.scratch.format import format_topic_xml
from uuid_extensions import uuid7str
from portkey_ai import AsyncPortkey
import re
import mermaid as md
from mermaid.graph import Graph

from s3 import download_model_list_from_s3
from langchain.docstore.document import Document

In [3]:
bedrock_client = AsyncPortkey(
        virtual_key="aldeo-portkey-b-5cf75f",
        config={"cache": {"mode": "simple"}},
    )

groq_client = AsyncPortkey(
    virtual_key="groqcloud-c9a2c3",
    config={"cache": {"mode": "simple"}},
)
    

In [4]:
# test data
topics = [
    {
        "presentation_id": "018efa53-2169-7dc0-ad63-e7084a09afc4",
        "presentation_topic": "Exploring Cultural Contrasts: Japan vs. the USA - Insights for American Expats",
        "presentation_audience": "American expats living in Japan",
        "presentation_goal": "To inform and offer practical tips to American expats living in Japan about the cultural contrasts between Japan and the USA",
        "id": "066236b0-e1e0-77d4-8000-590e777d7d29",
        "name": "Cultural Influence on Help Seeking and Support from Professionals and Others"
    },
    {
        "presentation_id": "018efa53-2169-7dc0-ad63-e7084a09afc4",
        "presentation_topic": "Exploring Cultural Contrasts: Japan vs. the USA - Insights for American Expats",
        "presentation_audience": "American expats living in Japan",
        "presentation_goal": "To inform and offer practical tips to American expats living in Japan about the cultural contrasts between Japan and the USA",
        "id": "066236b0-e1e0-7967-8000-39e03b240974",
        "name": "Indirect Communication in Japanese Culture"
    },
    {
        "presentation_id": "018efa6a-2208-79d3-a905-958f91b06da7",
        "presentation_topic": "Antitrust pressure against Apple: History, Antitrust Arguments, and Apple's Counterarguments",
        "presentation_audience": "Tech enthusiasts and consumers",
        "presentation_goal": "Just inform the audience without a persuasive bias",
        "id": "06623708-33af-7e71-8000-883e8038cf64",
        "name": "Apple's Antitrust Allegations and Competition Concerns"
    },
    {
        "presentation_id": "018efa76-2369-7861-b1b7-592b3e5c275f",
        "presentation_topic": "The 7 Natural Wonders of the World",
        "presentation_audience": "Travelers and nature enthusiasts",
        "presentation_goal": "To inform the audience about the 7 natural wonders of the world, their locations, and what makes them special",
        "id": "0662373e-4a90-7df1-8000-eaae8e2f7505",
        "name": "Mount Everest - The Tallest Peak in the World"
    },
    {
        "presentation_id": "018efa76-2369-7861-b1b7-592b3e5c275f",
        "presentation_topic": "The 7 Natural Wonders of the World",
        "presentation_audience": "Travelers and nature enthusiasts",
        "presentation_goal": "To inform the audience about the 7 natural wonders of the world, their locations, and what makes them special",
        "id": "06623734-5459-7ea5-8000-1842ed1b986e",
        "name": "Youngest Volcano in Northern Hemisphere - Paricutin, Mexico"
    },
    {
        "presentation_id": "018efa80-4216-71c9-b37c-b55cb8e17a2d",
        "presentation_topic": "Understanding the Transformer Model Architecture: Revolutionizing Natural Language Processing\n\nTo give an informative high-level overview of the Transformer model architecture.",
        "presentation_audience": "Artificial Intelligence and Machine Learning researchers and students",
        "presentation_goal": "To give an informative high-level overview of the Transformer model architecture.",
        "id": "0662375b-e9fe-7154-8000-0e67ffd84570",
        "name": "Multi-Head Attention Mechanisms in Transformer Architecture"
    }
]

In [5]:
slides = []

In [6]:
topic = topics[3]

In [7]:
all_docs = await download_model_list_from_s3(Document, f"{topic['presentation_id']}/all_sources_chunked_docs_with_topics.json")
reduced_docs = await reduce_docs(all_docs, 4000, topic)

Original character length of topic docs: 5710
Original number of topic docs: 14
Original number of sources: 4

Character length of topic docs after reduction by rerank score: 3997
Number of topic docs after reduction by rerank score: 10
Number of sources after reduction by rerank score: 4
Minimum rerank score increased from 0.4555922 to 0.8842394


In [8]:
# Add uuidv7 to each doc
for doc in reduced_docs:
    doc["metadata"]["id"] = uuid7str()

In [9]:
excerpt_ids = [doc['metadata']['id'] for doc in reduced_docs]
def get_id_from_suffix(suffix):
    return next((id for id in excerpt_ids if id.endswith(suffix)), None)

In [10]:
print(format_topic_xml(reduced_docs, show_sources=True))

<source>
<source_title>What are the 7 Natural Wonders of the World? - Exoticca Blog</source_title>

<excerpt>
<excerpt_id>07cd851e</excerpt_id>
<excerpt_headings>Mount Everest, Nepal, Tibet & China</excerpt_headings>
<excerpt_content>
Undoubtedly, Mount Everest is the most revered and legendary 7 natural wonders of the world. The tallest mountain on earth, Everest holds a mythical value. Many climbers have managed to summit the peak, but many more have been unsuccessful. Nestled in the heart of the Himalayas, Mount Everest measures 8,849 meters at its peak. The Nepali name for Everest is Sagarmatha, whilst in Tibet, it is known as Chomolungma, which roughly translates to ‘Mother Goddess of the World’.
</excerpt_content>
</excerpt>

</source>

<source>
<source_title>The 7 Natural Wonders Of The World - WorldAtlas</source_title>

<excerpt>
<excerpt_id>0fa56955</excerpt_id>
<excerpt_headings>Mount Everest</excerpt_headings>
<excerpt_content>
Mount Everest, the highest peak on Earth at 8,8

In [11]:
def get_presentation_xml(topic):
    return f"""<presentation>
    <topic>{topic['presentation_topic']}</topic>
    <audience>{topic['presentation_audience']}</audience>
    <goal>{topic['presentation_goal']}</goal>
</presentation>"""

In [12]:
print(get_presentation_xml(topic))

<presentation>
    <topic>The 7 Natural Wonders of the World</topic>
    <audience>Travelers and nature enthusiasts</audience>
    <goal>To inform the audience about the 7 natural wonders of the world, their locations, and what makes them special</goal>
</presentation>


In [13]:
slide_ideas_prompt = f"""\
I'm creating the presentation described below:
{get_presentation_xml(topic)}

I'm working on a section about <subtopic>{topic['name']}</subtopic> and I need some ideas for slides. Here's the research I've done for the section:
<research>
{format_topic_xml(reduced_docs, show_sources=False, show_excerpt_ids=False)}
</research>

Carefully read my research and devise a plan to create 1-4 powerpoint slides for the <subtopic></subtopic>. Here are the allowed slide types:

short_bullets_slide: 2-4 short unnested bullet points (max ~10 words per bullet)
long_bullets_slide: allows for more bullet points, longer bullet points, and nesting bullet points
markdown_table_slide: any table that can be written in markdown and fits reasonably on a slide
simple_mermaid_slide: a simple flow chart or diagram that can be expressed in mermaid, maximum 6 nodes.

Don't actually create the slide content in your plan. Instead, give me a list of <slide></slide>, each with a <slide_type></slide_type>, <slide_title></slide_title> and a concise but detailed <slide_content_idea></slide_content_idea in sentences of what information could be included in the slide.
Do not create an Introduction or Conclusion slide.
Make sure you consider not just the <subtopic></subtopic>, but also the <topic></topic>, the <audience></audience> and the <goal></goal> of the <presentation></presentation> I mentioned earlier when creating the slides."""

In [14]:
response = await bedrock_client.chat.completions.create(
            model="anthropic.claude-3-haiku-20240307-v1:0",
            # model="mistral.mistral-7b-instruct-v0:2",
            messages=[{"role": "user", "content": slide_ideas_prompt}],
            temperature=0,
            max_tokens=1500,
        )
response_text = response.choices[0].message.content.strip()

In [15]:
print(response_text)

Here is a plan for 4 slides for the "Mount Everest - The Tallest Peak in the World" section:

<slide>
<slide_type>short_bullets_slide</slide_type>
<slide_title>Mount Everest - The Tallest Peak</slide_title>
<slide_content_idea>
- Highest mountain on Earth at 8,849 meters (29,029 feet)
- Straddles the China (Tibet) and Nepal border
- Formed over 60 million years ago from metamorphic and sedimentary rocks
- Known as Sagarmatha in Nepal and Chomolungma in Tibet, meaning "Mother Goddess of the World"
</slide_content_idea>
</slide>

<slide>
<slide_type>long_bullets_slide</slide_type>
<slide_title>Climbing Mount Everest</slide_title>
<slide_content_idea>
- Extremely challenging and dangerous climb
- Harsh, unpredictable weather with temperatures below -60°C (-76°F)
- Around 800 climbers attempt to summit Everest each year
- The summit is 1 mile away but takes 12 hours on average to reach
- Climbing Everest requires specialized equipment, training, and experience
- Sherpas play a crucial role

In [16]:
llama_response = await groq_client.chat.completions.create(
    model="llama3-8b-8192",
    messages=[{"role": "user", "content": slide_ideas_prompt}],
    max_tokens=1500,
)
llama_response_text = llama_response.choices[0].message.content.strip()

In [18]:
print(llama_response_text)

Based on your research, I've created a plan for 4 slides for the "Mount Everest - The Tallest Peak in the World" subtopic. Here's the plan:

1. **slide_key**: simple_mermaid_slide
**slide_title**: Overview of Mount Everest
**slide_content_idea**: A simple diagram illustrating the location of Mount Everest, showing its bordering countries, and highlighting its geological composition.

2. **slide_key**: long_bullets_slide
**slide_title**: Interesting Facts About Mount Everest
**slide_content_idea**: A list of bullet points highlighting the height, location, and unique features of Mount Everest, such as its geological composition, weather conditions, and wildlife adapted to extreme altitudes.

3. **slide_key**: markdown_table_slide
**slide_title**: Comparison with Mauna Kea
**slide_content_idea**: A table comparing Mount Everest with Mauna Kea in terms of height, location, and geological composition, highlighting which one is technically the highest peak.

4. **slide_key**: short_bullets_

In [304]:
def parse_slides(input_string):
    slides = []
    slide_pattern = re.compile(r'<slide>(.*?)</slide>', re.DOTALL)
    slide_matches = slide_pattern.findall(input_string)

    for slide_match in slide_matches:
        slide_dict = {}
        
        slide_type_pattern = re.compile(r'<slide_type>(.*?)</slide_type>')
        slide_type_match = slide_type_pattern.search(slide_match)
        if slide_type_match:
            slide_dict['slide_type'] = slide_type_match.group(1)
        
        slide_title_pattern = re.compile(r'<slide_title>(.*?)</slide_title>')
        slide_title_match = slide_title_pattern.search(slide_match)
        if slide_title_match:
            slide_dict['slide_title'] = slide_title_match.group(1)
        
        slide_content_idea_pattern = re.compile(r'<slide_content_idea>(.*?)</slide_content_idea>', re.DOTALL)
        slide_content_idea_match = slide_content_idea_pattern.search(slide_match)
        if slide_content_idea_match:
            content_idea = slide_content_idea_match.group(1).strip()
            slide_dict['slide_content_idea'] = content_idea
        
        slides.append(slide_dict)
    
    return slides

slide_ideas = parse_slides(response_text)

In [305]:
slide_ideas

[{'slide_type': 'short_bullets_slide',
  'slide_title': 'Mount Everest - The Tallest Peak',
  'slide_content_idea': '- Highest mountain on Earth at 8,849 meters (29,029 feet)\n- Straddles the China (Tibet) and Nepal border\n- Formed over 60 million years ago from metamorphic and sedimentary rocks\n- Known as Sagarmatha in Nepal and Chomolungma in Tibet'},
 {'slide_type': 'long_bullets_slide',
  'slide_title': 'Climbing Mount Everest',
  'slide_content_idea': '- Approximately 800 climbers attempt to summit Everest each year\n- The climb to the summit is about 1 mile but takes 12 hours on average\n- Climbers face harsh and unpredictable weather, often below -60°C (-76°F)\n- Successful summits are challenging due to the extreme altitude and environmental conditions\n- Local Sherpa communities rely heavily on mountaineering tourism, raising ethical concerns\n- Environmental issues like melting glaciers and expedition waste are growing concerns'},
 {'slide_type': 'markdown_table_slide',
  '

In [332]:
def get_short_bullets_slide_prompt(presentation_xml, subtopic_name, slide_title, slide_content_idea):
    return f"""\
I'm creating the presentation described below:
{presentation_xml}

Here's the subtopic for the section I'm working on right now:
<subtopic>{subtopic_name}</subtopic>

Here's the research I have for the <subtopic></subtopic>:
<research>
{format_topic_xml(reduced_docs, show_sources=False, show_excerpt_ids=True)}
</research>

Within the <subtopic></subtopic>, I'm working on a slide called <slide_title>{slide_title}</slide_title>.
I need you to create 1-5 short bullet points to go on this slide. Each bullet point should be less than 10 words long and the bullet points cannot be nested.

Here's an idea for the slide content, to get you started:
<slide_content_idea>{slide_content_idea}</slide_content_idea>
You don't have to follow this idea exactly, but it should give you a sense of what I'm looking for.
You may only use the research I've provided to create the bullet points. Never include your background knowledge or other research.
You must include the <source_excerpt_id></source_excerpt_id> of the excerpt from the research above that supports the bullet point.

Give me the bullet points in the following XML format:
<bullet_points>
<bullet_point>
<content>The content of the bullet point</content>
<source_excerpt_id>a1b2c3d4</source_excerpt_id>
</bullet_point>
...
</bullet_points>

Finally, tell me whether or not the bullet points have a sequential order, like steps in a process or a timeline.
If they do, provide the order in the following XML format:
<sequential_order>yes</sequential_order>
If they do not, provide the following XML format:
<sequential_order>no</sequential_order>

Make sure you consider not just the <slide_title></slide_title>, but also the <subtopic></subtopic>, and the <topic></topic>, <audience></audience> and <goal></goal> of the <presentation></presentation> I mentioned earlier when creating the bullet points.
Remember you must cite an excerpt id as a source for every bullet point, so you can only use information found in my research."""

In [335]:
response = await bedrock_client.chat.completions.create(
            model="anthropic.claude-3-haiku-20240307-v1:0",
            # model="mistral.mistral-7b-instruct-v0:2",
            messages=[{"role": "user", "content": get_short_bullets_slide_prompt(get_presentation_xml(topic), topic['name'], slide_ideas[0]['slide_title'], slide_ideas[0]['slide_content_idea'])}],
            temperature=0,
            max_tokens=1500,
            stop=["</sequential_order>"],
        )
response_text = response.choices[0].message.content.strip() + "</sequential_order>"

In [336]:
print(response_text)

<bullet_points>
<bullet_point>
<content>Highest mountain on Earth at 8,849 meters (29,029 feet)</content>
<source_excerpt_id>2037c483</source_excerpt_id>
</bullet_point>
<bullet_point>
<content>Straddles the China (Tibet) and Nepal border</content>
<source_excerpt_id>e07a187e</source_excerpt_id>
</bullet_point>
<bullet_point>
<content>Formed over 60 million years ago from metamorphic and sedimentary rocks</content>
<source_excerpt_id>e07a187e</source_excerpt_id>
</bullet_point>
<bullet_point>
<content>Known as Sagarmatha in Nepal and Chomolungma in Tibet</content>
<source_excerpt_id>2462ba47</source_excerpt_id>
</bullet_point>
<bullet_point>
<content>Hosts unique wildlife like snow leopards and Himalayan tahr</content>
<source_excerpt_id>a291cedb</source_excerpt_id>
</bullet_point>
</bullet_points>

<sequential_order>no</sequential_order>


In [337]:
def parse_bullet_points(input_string):
    bullet_points = []
    sequential = False

    # Regular expression pattern to match the bullet_points XML
    pattern = r'<bullet_points>(.*?)</bullet_points>'
    match = re.search(pattern, input_string, re.DOTALL)

    if match:
        xml_string = match.group(1)

        # Regular expression pattern to match each bullet_point
        bullet_point_pattern = r'<bullet_point>(.*?)</bullet_point>'
        bullet_point_matches = re.findall(bullet_point_pattern, xml_string, re.DOTALL)

        for bullet_point_match in bullet_point_matches:
            content_pattern = r'<content>(.*?)</content>'
            source_excerpt_id_pattern = r'<source_excerpt_id>(.*?)</source_excerpt_id>'

            content_match = re.search(content_pattern, bullet_point_match, re.DOTALL)
            source_excerpt_id_match = re.search(source_excerpt_id_pattern, bullet_point_match, re.DOTALL)

            if content_match and source_excerpt_id_match:
                content = content_match.group(1).strip()
                source_excerpt_id_suffix = source_excerpt_id_match.group(1).strip()
                source_excerpt_id = get_id_from_suffix(source_excerpt_id_suffix)

                bullet_points.append({
                    'content': content,
                    'source_excerpt_id': source_excerpt_id
                })

        # Check for the sequential_order tag
        sequential_order_pattern = r'<sequential_order>(.*?)</sequential_order>'
        sequential_order_match = re.search(sequential_order_pattern, input_string, re.DOTALL)

        if sequential_order_match:
            sequential_order = sequential_order_match.group(1).strip().lower()
            if sequential_order == 'yes':
                sequential = True

    return bullet_points, sequential

bullet_points, sequential = parse_bullet_points(response_text)
slides.append({
    'slide_title': slide_ideas[0]['slide_title'],
    'slide_type': 'short_bullets_slide',
    'slide_content': bullet_points,
    'sequential': sequential
})

In [310]:
def get_long_bullet_points_slide_prompt(presentation_xml, subtopic_name, slide_title, slide_content_idea):
    return f"""\
I'm creating the presentation described below:
{presentation_xml}

Here's the subtopic for the section I'm working on right now:
<subtopic>{subtopic_name}</subtopic>

Here's the research I have for the <subtopic></subtopic>:
<research>
{format_topic_xml(reduced_docs, show_sources=False, show_excerpt_ids=True)}
</research>

Within the <subtopic></subtopic>, I'm working on a slide called <slide_title>{slide_title}</slide_title>.
I need you to create the bullet point content for this slide. 

Here's an idea for the slide content, to get you started:
<slide_content_idea>{slide_content_idea}</slide_content_idea>
You don't have to follow this idea exactly, but it should give you a sense of what I'm looking for.
You may only use the research I've provided to create the bullet points. Never include your background knowledge or other research.
For each bullet point you include, you must use a footnote to cite the <excerpt_id></excerpt_id> of the excerpt from the research above that supports the bullet point.

Use markdown to format the bullet points. You can use numbered and unnumbered lists, and you should nest lists whenever possible. You should also use other markdown conventions like bold and italics when appropriate. Seperate the bullet points and the footnotes with a horizontal rule (---).

Give me the bullet point content in markdown, with footnotes enclosed in a <content></content> tag, like this:
<content>
- The content of the bullet point[^1]
    - The content of another bullet point[^2]

---
[^1]: <excerpt_id>a1b2c3d4</excerpt_id>
[^2]: <excerpt_id>e5f6g7h8</excerpt_id>
</content>

Make sure you consider not just the <slide_title></slide_title>, but also the <subtopic></subtopic>, and the <topic></topic>, <audience></audience> and <goal></goal> of the <presentation></presentation> I mentioned earlier when creating the bullet points.
Remember you must cite an excerpt id as a source for every bullet point, so you can only use information found in my research."""

In [311]:
response = await bedrock_client.chat.completions.create(
    model="anthropic.claude-3-haiku-20240307-v1:0",
    # model="mistral.mistral-7b-instruct-v0:2",
    messages=[{"role": "user", "content": get_long_bullet_points_slide_prompt(get_presentation_xml(topic), topic['name'], slide_ideas[1]['slide_title'], slide_ideas[1]['slide_content_idea'])}],
    temperature=0,
    max_tokens=1500,
    stop=["</content>"],
)
response_text = response.choices[0].message.content.strip() + "\n</content>"

In [312]:
print(response_text)

<content>
- Approximately 800 climbers attempt to summit Everest each year[^1]
- The climb to the summit is about 1 mile but takes 12 hours on average[^2]
- Climbers face harsh and unpredictable weather, often below -60°C (-76°F)[^3]
- Successful summits are challenging due to the extreme altitude and environmental conditions[^4]
- Local Sherpa communities rely heavily on mountaineering tourism, raising ethical concerns[^5]
- Environmental issues like melting glaciers and expedition waste are growing concerns[^5]

---
[^1]: <excerpt_id>3e37b7dd</excerpt_id>
[^2]: <excerpt_id>3e37b7dd</excerpt_id>
[^3]: <excerpt_id>25f045d2</excerpt_id>
[^4]: <excerpt_id>1060b915</excerpt_id>
[^5]: <excerpt_id>a291cedb</excerpt_id>
</content>


In [313]:
def parse_content_and_footnotes(input_string):
    # Pattern to match the content before the horizontal rule, excluding <content> tags
    content_pattern = r'<content>\s*(.*?)(?:---|\*\*\*|\+\+\+|===|___)\s*'
    content_match = re.search(content_pattern, input_string, re.DOTALL)
    content = content_match.group(1).strip() if content_match else ''

    # Pattern to match the footnotes
    footnote_pattern = r'\[\^(\d+)\]:\s*<excerpt_id>(\w+)</excerpt_id>'
    footnote_matches = re.findall(footnote_pattern, input_string)

    footnotes = {num: get_id_from_suffix(excerpt_id) for num, excerpt_id in footnote_matches}

    return {
        'content': content,
        'footnotes': footnotes
    }

parsed_content_and_footnotes = parse_content_and_footnotes(response_text)
slides.append({
    'slide_title': slide_ideas[1]['slide_title'],
    'slide_type': 'long_bullets_slide',
    'slide_content': parsed_content_and_footnotes
})

In [325]:
# Here's an idea for the slide content, to get you started:
# <slide_content_idea>{slide_content_idea}</slide_content_idea>

# You don't have to follow this idea exactly, but it should give you a sense of what I'm looking for.

def get_markdown_table_slide_prompt(presentation_xml, subtopic_name, slide_title, slide_content_idea):
    return f"""\
I'm creating the presentation described below:
{presentation_xml}

Here's the subtopic for the section I'm working on right now:
<subtopic>{subtopic_name}</subtopic>

Here's the research I have for the <subtopic></subtopic>:
<research>
{format_topic_xml(reduced_docs, show_sources=False, show_excerpt_ids=True)}
</research>

Within the <subtopic></subtopic>, I'm working on a slide called <slide_title>{slide_title}</slide_title>.
I need you to create a markdown table for this slide.

Here's an idea for the slide content, to get you started:
<slide_content_idea>{slide_content_idea}</slide_content_idea>

You don't have to follow this idea exactly, but it should give you a sense of what I'm looking for.

You may only use the research I've provided to create the table. Never include your background knowledge or other research.
IMPORTANT: For every fact you include in the table, you must use a footnote to cite the <excerpt_id></excerpt_id> of the excerpt from the research above that supports the fact.

Use markdown to format the table and the footnotes. You should use other markdown conventions like lists, bold and italics within the table cells. Seperate the table and the footnotes with a horizontal rule (---).

Give me the markdown table content in markdown, with footnotes all enclosed in a <content></content> tag, like this:
<content>
| Header 1 | Header 2 | Header 3 |
|----------|----------|----------|
| Content 1[^2] | Content 2[^1] | Content 3[^3] |
| Content 4[^4] | Content 5[^3] | - Content 6[^5]\n-Content 7[^5] |

---
[^1]: <excerpt_id>a1b2c3d4</excerpt_id>
[^2]: <excerpt_id>e5f6g7h8</excerpt_id>
[^3]: <excerpt_id>i9j0k1l2</excerpt_id>
[^4]: <excerpt_id>q7r8s9t0</excerpt_id>
[^5]: <excerpt_id>u1v2w3x4</excerpt_id>
</content>

Make sure you consider not just the <slide_title></slide_title>, but also the <subtopic></subtopic>, and the <topic></topic>, <audience></audience> and <goal></goal> of the <presentation></presentation> I mentioned earlier when creating the table.
Remember you must cite an excerpt id as a source for every fact in the table, so you can only use information found in my research."""

In [326]:
response = await bedrock_client.chat.completions.create(
    model="anthropic.claude-3-haiku-20240307-v1:0",
    # model="mistral.mistral-7b-instruct-v0:2",
    messages=[{"role": "user", "content": get_markdown_table_slide_prompt(get_presentation_xml(topic), topic['name'], slide_ideas[2]['slide_title'], slide_ideas[2]['slide_content_idea'])}],
    temperature=0,
    max_tokens=1500,
    stop=["</content>"],
)
response_text = response.choices[0].message.content.strip() + "\n</content>"
print(response_text)

<content>
| Fact | Detail |
| --- | --- |
| Height | 8,849 meters (29,029 feet)[^1] |
| Location | Himalayan mountain range, Nepal and Tibet (China) border[^1] |
| Local Names | Sagarmatha (Nepal), Chomolungma (Tibet)[^2] |
| First Ascent | 1953 by Edmund Hillary and Tenzing Norgay[^3] |
| Wildlife | Snow leopard, Himalayan tahr[^4] |
| Environmental Significance | Source of major Asian rivers, melting glaciers[^4][^5] |

---
[^1]: <excerpt_id>e07a187e</excerpt_id>
[^2]: <excerpt_id>2462ba47</excerpt_id>
[^3]: <excerpt_id>3e37b7dd</excerpt_id>
[^4]: <excerpt_id>a291cedb</excerpt_id>
[^5]: <excerpt_id>25f045d2</excerpt_id>
</content>


In [316]:
response_text

'<content>\n| Mount Everest Facts |\n|---------------------|\n| **Height**: 8,849 meters (29,029 feet) above sea level[^1] |\n| **Other Names**: Sagarmatha (Nepal), Chomolungma (Tibet)[^2] |\n| **First Ascent**: Conquered in 1953 by Sir Edmund Hillary and Tenzing Norgay[^3] |\n| **Climbing Difficulty**: Extremely challenging due to harsh weather, thin air, and technical climbing[^4] |\n| **Wildlife**: Home to snow leopards and Himalayan tahr[^5] |\n| **Environmental Concerns**: Melting glaciers and expedition waste have raised sustainability issues[^5] |\n| **Climbing Attempts**: Around 800 climbers attempt to summit Everest yearly[^6] |\n\n---\n[^1]: <excerpt_id>e07a187e</excerpt_id>\n[^2]: <excerpt_id>2462ba47</excerpt_id>\n[^3]: <excerpt_id>3e37b7dd</excerpt_id>\n[^4]: <excerpt_id>25f045d2</excerpt_id>\n[^5]: <excerpt_id>a291cedb</excerpt_id>\n[^6]: <excerpt_id>3e37b7dd</excerpt_id>\n</content>'

In [317]:

def parse_table_and_footnotes(input_string):
    # Pattern to match the content before the horizontal rule, excluding <content> tags
    content_pattern = r'<content>\s*(.*?)(?:\n---|\n\*\*\*|\n\+\+\+|\n===|\n___)\s*'
    content_match = re.search(content_pattern, input_string, re.DOTALL)
    table = content_match.group(1).strip() if content_match else ''

    # Pattern to match the footnotes
    footnote_pattern = r'\[\^(\d+)\]:\s*<excerpt_id>(\w+)</excerpt_id>'
    footnote_matches = re.findall(footnote_pattern, input_string)
    footnotes = {num: get_id_from_suffix(excerpt_id) for num, excerpt_id in footnote_matches}

    return {
        'table': table,
        'footnotes': footnotes
    }
    
parsed_content_and_footnotes = parse_table_and_footnotes(response_text)
slides.append({
    'slide_title': slide_ideas[2]['slide_title'],
    'slide_type': 'markdown_table_slide',
    'slide_content': parsed_content_and_footnotes
})

In [318]:
print(parsed_content_and_footnotes['table'])

| Mount Everest Facts |
|---------------------|
| **Height**: 8,849 meters (29,029 feet) above sea level[^1] |
| **Other Names**: Sagarmatha (Nepal), Chomolungma (Tibet)[^2] |
| **First Ascent**: Conquered in 1953 by Sir Edmund Hillary and Tenzing Norgay[^3] |
| **Climbing Difficulty**: Extremely challenging due to harsh weather, thin air, and technical climbing[^4] |
| **Wildlife**: Home to snow leopards and Himalayan tahr[^5] |
| **Environmental Concerns**: Melting glaciers and expedition waste have raised sustainability issues[^5] |
| **Climbing Attempts**: Around 800 climbers attempt to summit Everest yearly[^6] |


In [327]:
# Here's an idea for the mermaid diagram, to get you started:
# <mermaid_diagram_idea>
# {slide_content_idea}
# </mermaid_diagram_idea>
# You don't have to follow this idea exactly, but it should give you a sense of what I'm looking for.


def get_simple_mermaid_slide_prompt(presentation_xml, subtopic_name, slide_title, slide_content_idea):
    return f"""\
I'm creating the presentation described below:
{presentation_xml}

Here's the subtopic for the section I'm working on right now:
<subtopic>{subtopic_name}</subtopic>

Here's the research I have for the <subtopic></subtopic>:
<research>
{format_topic_xml(reduced_docs, show_sources=False, show_excerpt_ids=True)}
</research>

Within the <subtopic></subtopic>, I'm working on a slide called <slide_title>{slide_title}</slide_title>.
I need you to create a mermaid diagram for this slide.

You may only use the research I've provided to create the diagram. Never include your background knowledge or other research.
After you create the diagram, you must create a list of <excerpt_id></excerpt_id> to cite every excerpt from the research that supports the facts and information you included in the diagram.

Give me the mermaid diagram content, as well as the list of excerpt ids in the following xml format:
<mermaid_diagram>
flowchart TD
    A[Christmas] -->|Get money| B(Go shopping)
    B --> C{{Let me think}}
    C -->|One| D[Laptop]
    C -->|Two| E[iPhone]
    C -->|Three| F[fa:fa-car Car]
</mermaid_diagram>
<sources>
<excerpt_id>a1b2c3d4</excerpt_id>
<excerpt_id>e5f6g7h8</excerpt_id>
<excerpt_id>i9j0k1l2</excerpt_id>
</sources>

It's imperative that you use correct mermaid syntax so that the diagram is rendered correctly without any errors.
Remember to cite an excerpt id as a source for every fact in the diagram, and only use information found in my research."""


In [328]:
response = await bedrock_client.chat.completions.create(
    model="anthropic.claude-3-haiku-20240307-v1:0",
    # model="mistral.mistral-7b-instruct-v0:2",
    messages=[{"role": "user", "content": get_simple_mermaid_slide_prompt(get_presentation_xml(topic), topic['name'], slide_ideas[3]['slide_title'], slide_ideas[3]['slide_content_idea'])}],
    temperature=0,
    max_tokens=1500,
    stop=["</sources>"],
)
response_text = response.choices[0].message.content.strip() + "</sources>"

In [329]:
def clean_mermaid_code(code):
    # Replace double quotes with single quotes inside node text
    code = re.sub(r'(\w+\[|{|\()([^"\]]+)"([^"\]]+)"(?=\])', r'\1\2\'\3\'', code)
    
    # Wrap node text with double quotes, handling nested parentheses
    code = re.sub(r'(\w+\[)([^"\[\]\(\)\{\}]+(?:\([^)]*\))*[^"\[\]\(\)\{\}]*)(\])', r'\1"\2"\3', code)
    
    return code

def parse_mermaid_and_sources(input_string):
    # Pattern to match the mermaid diagram code
    mermaid_pattern = r'<mermaid_diagram>\s*(.*?)\s*</mermaid_diagram>'
    mermaid_match = re.search(mermaid_pattern, input_string, re.DOTALL)
    mermaid_code = mermaid_match.group(1).strip() if mermaid_match else ''
    mermaid_code_cleaned = clean_mermaid_code(mermaid_code)

    # Pattern to match the excerpt IDs
    excerpt_pattern = r'<excerpt_id>(.*?)</excerpt_id>'
    excerpt_id_suffixes = re.findall(excerpt_pattern, input_string)
    excerpt_ids = [get_id_from_suffix(suffix) for suffix in excerpt_id_suffixes]

    return {
        'mermaid_code': mermaid_code_cleaned,
        'excerpt_ids': excerpt_ids
    }

parsed_mermaid_and_sources = parse_mermaid_and_sources(response_text)
slides.append({
    'slide_title': slide_ideas[3]['slide_title'],
    'slide_type': 'simple_mermaid_slide',
    'slide_content': parsed_mermaid_and_sources
})

In [330]:
print(parsed_mermaid_and_sources['mermaid_code'])

flowchart TD
    A["Plate Tectonics"] --> B(Indian Plate)
    B --> C{Collision with Eurasian Plate}
    C --> D["Uplift of Himalayan Mountains"]
    D --> E["Formation of Mount Everest"]
    E --> F["Highest Peak on Earth"]


In [331]:
graph: Graph = Graph('example-flowchart', parsed_mermaid_and_sources['mermaid_code'])
graphe: md.Mermaid = md.Mermaid(graph)
graphe # !! note this work just in notbooke that render html.

In [338]:
slides

[{'slide_title': 'Mount Everest - The Tallest Peak',
  'slide_type': 'short_bullets_slide',
  'slide_content': [{'content': 'Highest mountain on Earth at 8,849 meters (29,029 feet)',
    'source_excerpt_id': '06627945-7ad9-7b2c-8000-29dc2037c483'},
   {'content': 'Straddles the China (Tibet) and Nepal border',
    'source_excerpt_id': '06627945-7ad9-7a73-8000-5963e07a187e'},
   {'content': 'Formed over 60 million years ago from metamorphic and sedimentary rocks',
    'source_excerpt_id': '06627945-7ad9-7a73-8000-5963e07a187e'},
   {'content': 'Known as Sagarmatha in Nepal and Chomolungma in Tibet',
    'source_excerpt_id': '06627945-7ad9-7aa6-8000-390d2462ba47'},
   {'content': 'Harsh and unpredictable weather, often below -60°C (-76°F)',
    'source_excerpt_id': '06627945-7ad9-7ae9-8000-d56525f045d2'}]},
 {'slide_title': 'Climbing Mount Everest',
  'slide_type': 'long_bullets_slide',
  'slide_content': {'content': '- Approximately 800 climbers attempt to summit Everest each year[^1]\n